# TAARHOGEN2 — First Generation Runtime

Fresh engine, independent of TAARHOGEN1 and its CLIP assets. This notebook installs the repository code and attempts local GPU inference with FLUX.2 Klein 4B.

In [ ]:
import torch, platform, shutil, os
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('STOP: No CUDA GPU is attached. In Colab choose Runtime > Change runtime type > T4 GPU, reconnect, then rerun this cell. No model files have been downloaded yet.')
props = torch.cuda.get_device_properties(0)
vram_gib = props.total_memory / 1024**3
free_disk_gib = shutil.disk_usage('/content').free / 1024**3
try:
    import psutil
    ram_gib = psutil.virtual_memory().total / 1024**3
except Exception:
    ram_gib = None
print('GPU:', props.name)
print('Total VRAM GiB:', round(vram_gib, 2))
print('Free disk GiB:', round(free_disk_gib, 2))
print('System RAM GiB:', round(ram_gib, 2) if ram_gib else 'unavailable')
print('BF16 supported:', torch.cuda.is_bf16_supported())
if vram_gib < 13:
    print('WARNING: Official model card reports about 13 GB VRAM. CPU offload may help, but this GPU is below the stated target; generation may fail.')
if free_disk_gib < 25:
    print('WARNING: Less than 25 GB free disk. Model download/cache may run out of space.')
print('PREFLIGHT_COMPLETE')


In [ ]:
%cd /content
!git clone --branch dev --single-branch https://github.com/taahro/Taarhogen2.git
%cd /content/Taarhogen2
!pip install -q -r requirements.txt
print('REPOSITORY_AND_DEPENDENCIES_READY')


## Generate the first image

The official FLUX.2 Klein 4B model card reports approximately 13 GB VRAM and documents `Flux2KleinPipeline` with CPU offload, 4 steps and guidance 1.0. The preflight cell reports actual hardware before any model download. This is local GPU inference only—no hosted API or paid endpoint.


In [ ]:
from taarhogen2.generate import generate_image
from taarhogen2.studio import StudioSettings

settings = StudioSettings(
    view='VIEW 01',
    focal_length_mm=50,
    aperture=2.8,
    lighting='soft studio',
    environment='studio',
    finish='natural',
)

output = generate_image(
    prompt='A premium editorial photograph of a sculptural glass object on a dark studio plinth, precise reflections, rich material detail',
    output='outputs/taarhogen2-first.png',
    width=1024,
    height=1024,
    steps=4,
    guidance=1.0,
    seed=424242,
    settings=settings,
)
print('PNG_READY:', output)

In [ ]:
from IPython.display import display
from PIL import Image
image = Image.open('outputs/taarhogen2-first.png')
print('Format:', image.format, '| Size:', image.size)
display(image)